# Task 4: 네이버와 구글의 서버 헤더 정보 비교하기

## 과제 목표
- `requests` 모듈을 사용하여 **네이버(naver.com)**와 **구글(google.com)**의 HTTP 응답 헤더를 수집한다.
- 두 서버의 헤더 정보를 **비교 분석**하여 서버 아키텍처 차이를 이해한다.

## 학습 목표
1. HTTP 헤더(Header)의 개념과 역할을 이해한다.
2. `requests.head()` 메서드의 동작 원리를 파악한다.
3. 서버가 반환하는 헤더 필드별 의미를 학습한다.
4. 두 글로벌 서비스의 서버 설정 차이를 분석한다.

---
## 1단계: 환경 설정 및 라이브러리 임포트

### 왜 `requests` 라이브러리를 사용하는가?

Python에서 HTTP 요청을 보내는 방법은 크게 두 가지입니다:

| 방법 | 장점 | 단점 |
|------|------|------|
| `urllib` (표준 라이브러리) | 별도 설치 불필요 | 코드가 복잡하고 직관적이지 않음 |
| `requests` (서드파티) | 직관적이고 간결한 API | pip 설치 필요 |

`requests`는 **"HTTP for Humans"**라는 슬로건답게, 복잡한 HTTP 통신을 단 한 줄로 처리할 수 있습니다.
실무에서도 90% 이상의 Python 개발자가 HTTP 요청 시 `requests`를 사용합니다.

### 왜 `pandas`도 함께 임포트하는가?

헤더 데이터는 **키-값 쌍(dictionary)**으로 반환됩니다. 이를 `pandas.DataFrame`으로 변환하면:
- 두 서버의 헤더를 **나란히 비교**할 수 있고
- Jupyter Notebook에서 **깔끔한 표 형태**로 렌더링됩니다.

In [ ]:
import requests
import pandas as pd

print(f"requests 버전: {requests.__version__}")
print(f"pandas 버전: {pd.__version__}")
print("환경 설정 완료!")

---
## 2단계: HTTP HEAD 요청으로 헤더 정보 수집

### HTTP 요청 메서드: HEAD vs GET

HTTP에는 여러 요청 메서드가 있습니다. 여기서 중요한 두 가지를 비교합니다:

| 메서드 | 동작 | 응답 본문(Body) | 용도 |
|--------|------|:---------------:|------|
| `GET` | 리소스 전체를 요청 | **포함** | 웹 페이지 로딩, API 데이터 수집 |
| `HEAD` | 헤더만 요청 | **미포함** | 서버 상태 확인, 메타데이터 수집 |

### 왜 `HEAD`를 사용하는가?

우리의 목적은 **서버 헤더 정보만** 얻는 것입니다. `GET`을 사용하면:
- 네이버/구글의 **전체 HTML 페이지**(수백 KB)를 불필요하게 다운로드
- 네트워크 트래픽 낭비 + 응답 시간 증가

`HEAD`는 **헤더만 반환**하므로 훨씬 가볍고 빠릅니다. 이는 실무에서 서버 상태를 점검할 때 자주 사용하는 기법입니다.

### `requests.head()` 함수의 파라미터

```python
requests.head(url, **kwargs)
```

- `url`: 요청할 URL (문자열)
- `allow_redirects=True`: 리다이렉트를 따라갈지 여부. HEAD 요청은 기본값이 `False`이지만, 네이버/구글은 HTTPS 리다이렉트가 있으므로 `True`로 설정합니다.
- `timeout=10`: 10초 내 응답이 없으면 에러 발생. 무한 대기를 방지하는 **방어적 프로그래밍** 기법입니다.

In [ ]:
# 네이버 서버 헤더 수집
naver_response = requests.head(
    "https://www.naver.com",
    allow_redirects=True,   # HTTPS 리다이렉트 허용
    timeout=10              # 10초 타임아웃 설정
)

# 구글 서버 헤더 수집
google_response = requests.head(
    "https://www.google.com",
    allow_redirects=True,
    timeout=10
)

print(f"네이버 응답 상태 코드: {naver_response.status_code}")
print(f"구글 응답 상태 코드: {google_response.status_code}")
print(f"\n네이버 헤더 개수: {len(naver_response.headers)}개")
print(f"구글 헤더 개수: {len(google_response.headers)}개")

---
## 3단계: 각 서버의 헤더 전체 내용 확인

### `response.headers`의 정체

`response.headers`는 Python의 일반 딕셔너리(`dict`)가 아닙니다.
**`CaseInsensitiveDict`**라는 특수 딕셔너리로, **키의 대소문자를 구분하지 않습니다.**

예를 들어:
```python
headers['Content-Type']  # OK
headers['content-type']  # 같은 결과!
headers['CONTENT-TYPE']  # 이것도 같은 결과!
```

이는 HTTP 표준(RFC 7230)에서 **헤더 필드명은 대소문자를 구분하지 않는다**고 정의했기 때문입니다.

### `dict()`로 변환하는 이유

`dict(response.headers)`로 변환하면 일반 딕셔너리가 되어:
- `pandas.DataFrame`으로 쉽게 변환 가능
- `.items()`, `.keys()`, `.values()` 등 표준 메서드 사용 가능

In [ ]:
# 네이버 헤더 전체 출력
print("=" * 60)
print("[네이버 서버 헤더]")
print("=" * 60)
naver_headers = dict(naver_response.headers)
for key, value in naver_headers.items():
    print(f"  {key}: {value}")

print("\n")

# 구글 헤더 전체 출력
print("=" * 60)
print("[구글 서버 헤더]")
print("=" * 60)
google_headers = dict(google_response.headers)
for key, value in google_headers.items():
    print(f"  {key}: {value}")

---
## 4단계: 두 서버의 헤더를 DataFrame으로 비교

### 비교 전략

두 서버의 헤더 필드명이 완전히 같지 않습니다. 따라서:
1. **양쪽 헤더의 모든 키를 합집합(union)**으로 수집
2. 각 키에 대해 네이버/구글 값을 매핑
3. 값이 없는 경우 `"(없음)"`으로 표시

### `pd.DataFrame.from_dict()` vs `pd.DataFrame()` 차이

| 방법 | 사용 시점 |
|------|----------|
| `pd.DataFrame(data)` | 리스트, 딕셔너리 등 일반적인 경우 |
| `pd.DataFrame.from_dict(data, orient='index')` | 딕셔너리의 키를 **행(index)**으로 사용할 때 |

여기서는 헤더 필드명을 행으로, 네이버/구글 값을 열로 배치하기 위해 `from_dict`에 `orient='index'`를 사용합니다.

In [ ]:
# 양쪽 헤더의 모든 키를 합집합으로 수집
all_keys = sorted(set(naver_headers.keys()) | set(google_headers.keys()))

# 비교 데이터 구성
comparison_data = {}
for key in all_keys:
    comparison_data[key] = {
        "네이버 (Naver)": naver_headers.get(key, "(없음)"),
        "구글 (Google)": google_headers.get(key, "(없음)")
    }

# DataFrame 생성 — orient='index'로 헤더 필드명을 행 인덱스로 사용
df_comparison = pd.DataFrame.from_dict(comparison_data, orient="index")
df_comparison.index.name = "헤더 필드"

print(f"총 {len(df_comparison)}개의 헤더 필드를 비교합니다.\n")
df_comparison

---
## 5단계: 주요 헤더 필드별 의미 분석

### HTTP 헤더란?

HTTP 헤더는 **클라이언트(브라우저)와 서버 사이에 주고받는 메타데이터**입니다.
편지에 비유하면, 편지 내용(Body)과 별도로 **봉투에 적힌 정보**(발신자, 우편번호, 등기 여부 등)에 해당합니다.

### 주요 헤더 필드의 의미

| 헤더 필드 | 의미 | 비유 |
|-----------|------|------|
| `Server` | 서버 소프트웨어 이름 | "이 편지는 ○○ 우체국에서 발송" |
| `Content-Type` | 응답 데이터의 형식 | "이 봉투 안에는 한국어 문서가 들어있음" |
| `Content-Length` | 응답 데이터의 크기(바이트) | "봉투 무게: 500g" |
| `Cache-Control` | 캐시 정책 | "이 편지는 7일 동안 보관해도 됨" |
| `X-Frame-Options` | iframe 삽입 허용 여부 | "이 편지를 다른 편지에 넣어서 보내지 마시오" |
| `Content-Encoding` | 압축 방식 | "이 편지는 gzip으로 압축되어 있음" |
| `Set-Cookie` | 쿠키 설정 | "다음에 올 때 이 출입증을 가져오세요" |
| `Strict-Transport-Security` | HTTPS 강제 | "앞으로는 반드시 등기(HTTPS)로만 보내세요" |

아래 코드에서 이 주요 필드들만 추출하여 상세 비교합니다.

In [ ]:
# 주요 헤더 필드 선별 비교
important_fields = [
    "Server", "Content-Type", "Content-Length", "Content-Encoding",
    "Cache-Control", "X-Frame-Options", "X-Content-Type-Options",
    "Strict-Transport-Security", "Set-Cookie", "Date"
]

important_data = []
for field in important_fields:
    naver_val = naver_headers.get(field, "(없음)")
    google_val = google_headers.get(field, "(없음)")
    
    # 값이 너무 길면 80자로 잘라서 표시
    if len(str(naver_val)) > 80:
        naver_val = str(naver_val)[:80] + "..."
    if len(str(google_val)) > 80:
        google_val = str(google_val)[:80] + "..."
    
    important_data.append({
        "헤더 필드": field,
        "네이버": naver_val,
        "구글": google_val,
        "동일 여부": "✅ 동일" if naver_val == google_val else "❌ 다름"
    })

df_important = pd.DataFrame(important_data)
print("=" * 60)
print("주요 헤더 필드 비교")
print("=" * 60)
df_important

---
## 6단계: 공통 헤더와 고유 헤더 분석

### 왜 공통/고유 헤더를 구분하는가?

- **공통 헤더**: HTTP 표준에서 정의한 필수/권장 필드. 대부분의 서버가 반환합니다.
- **네이버만의 헤더**: 네이버의 고유한 서버 설정이나 보안 정책을 반영합니다.
- **구글만의 헤더**: 구글의 자체 인프라(GFE, GWS 등)와 관련된 설정입니다.

### 집합(Set) 연산으로 분류하는 이유

Python의 `set`은 **집합 연산**(교집합 `&`, 차집합 `-`, 합집합 `|`)을 지원합니다.
이를 활용하면 두 딕셔너리의 키를 수학적으로 깔끔하게 분류할 수 있습니다.

In [ ]:
naver_keys = set(naver_headers.keys())
google_keys = set(google_headers.keys())

# 집합 연산으로 분류
common_keys = naver_keys & google_keys      # 교집합: 양쪽 다 있는 헤더
naver_only = naver_keys - google_keys        # 차집합: 네이버만 있는 헤더
google_only = google_keys - naver_keys       # 차집합: 구글만 있는 헤더

print(f"공통 헤더: {len(common_keys)}개")
print(f"  → {sorted(common_keys)}")

print(f"\n네이버만의 헤더: {len(naver_only)}개")
print(f"  → {sorted(naver_only)}")

print(f"\n구글만의 헤더: {len(google_only)}개")
print(f"  → {sorted(google_only)}")

---
## 핵심 정리

### 이번 과제에서 배운 것

| 개념 | 핵심 내용 |
|------|----------|
| **HTTP HEAD** | 헤더만 받아오는 경량 요청 메서드. 서버 정보 수집에 최적 |
| **response.headers** | `CaseInsensitiveDict` — 대소문자 무관하게 접근 가능 |
| **헤더 필드** | Server, Content-Type, Cache-Control 등이 서버의 보안·성능 정책을 결정 |
| **집합 연산** | `set`의 `&`, `-`, `|`로 두 딕셔너리의 키를 수학적으로 비교 |

### 네이버 vs 구글 핵심 차이

- **서버 소프트웨어**: 네이버는 자체 서버(NWS), 구글은 GWS(Google Web Server) 사용
- **보안 헤더**: 양쪽 모두 X-Frame-Options, HSTS 등 보안 헤더를 적용하지만 세부 설정이 다름
- **캐시 정책**: 구글은 적극적 캐싱, 네이버는 상대적으로 보수적인 캐시 전략 사용